# ADK 2.x Graph Workflows: Human-in-the-Loop

This notebook demonstrates how to build a workflow that requires human intervention before proceeding. 

**The Workflow:**
1. Takes a customer complaint.
2. An LLM agent drafts an email response.
3. **Execution Pauses:** The system yields a `RequestInput` event and waits for a human reviewer.
4. The human can reply with "approve", "reject", or provide text feedback.
5. If feedback is provided, the workflow loops back to the LLM to rewrite the draft.

### Configuration
Configure the environment to use Google Cloud Vertex AI and Application Default Credentials.

In [ ]:
import os

LOCATION = "us-central1"
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"  # Use Agent Platform API

In [ ]:
%%bash
echo > adk_agents/.env "GOOGLE_CLOUD_LOCATION=$GOOGLE_CLOUD_LOCATION
GOOGLE_GENAI_USE_VERTEXAI=$GOOGLE_GENAI_USE_VERTEXAI
"

### Imports
Notice the special import: `RequestInput`.

In [ ]:
from typing import Literal
from google.adk import Agent
from google.adk import Event
from google.adk import Workflow
from google.adk.events import RequestInput
from pydantic import BaseModel, Field

MODEL = "gemini-2.5-flash"

### Node: Process Input
Initializes the workflow state with the customer's complaint and an empty feedback string.

In [ ]:
def process_input(node_input: str):
    """Takes the initial customer complaint as input and sets it in the state."""
    yield Event(state={"complaint": node_input, "feedback": ""})

### Node: Draft Email (LLM Agent)
Drafts the email. It reads `{complaint}` from the state, and conditionally applies `{feedback?}` if the human reviewer sent it back for a rewrite. The result is stored in the state under `draft`.

In [ ]:
draft_email = Agent(
    name="draft_email",
    model=MODEL,
    instruction="""
    Please write a polite, helpful response email to the following customer complaint: "{complaint}"
    If there is any feedback from the manager to revise the draft, please incorporate it: "{feedback?}"
    """,
    output_key="draft",
)

### Node: Request Human Review
Instead of returning a standard `Event`, this node yields a `RequestInput` object. 

**This tells the ADK Runner to suspend execution**, save the state to the Session Service, and surface the message to the user.

In [ ]:
def request_human_review(draft: str):
    yield RequestInput(
        message=(
            f"""Please review the following draft "
            email and provide 'approve',
            'reject', or feedback to revise.
            \n\n---\n{draft}\n---"""
        ),
    )

### Node: Handle Human Review (Router)
When the workflow is resumed with a new user message, that message is passed directly into this node as `node_input`. This function dictates the next edge based on the human's response.

In [ ]:
def handle_human_review(node_input: str):
    if node_input.lower().strip() == "reject":
        yield Event(route="rejected")
    elif node_input.lower().strip() == "approve":
        yield Event(route="approved")
    else:
        # If they typed anything else, treat it as revision feedback.
        yield Event(state={"feedback": node_input}, route="revise")

### Terminal Nodes
Simple functions to output the final status of the email.

In [ ]:
def reject_email():
    yield Event(message="Draft rejected.")

def send_email(draft: str):
    yield Event(message="Draft approved and sent successfully.")

### Assemble the Workflow
Define the graph structure mapping the routing strings to the terminal nodes, or looping back to `draft_email`.

In [ ]:
root_agent = Workflow(
    name="request_input",
    edges=[
        (
            "START",
            process_input,
            draft_email,
            request_human_review,
            handle_human_review,
        ),
        (
            handle_human_review,
            {
                "revise": draft_email,
                "approved": send_email,
                "rejected": reject_email,
            },
        ),
    ],
)

### Execute - Phase 1: Trigger and Pause
We start the workflow. It will process the complaint, generate the draft, and then pause execution, waiting for our input.

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part
from google.adk.events import RequestInput

session_service = InMemorySessionService()
session = await session_service.create_session(
    app_name="hitl_app", 
    user_id="human_reviewer",
    session_id="session_01"
)
runner = Runner(agent=root_agent, app_name="hitl_app", session_service=session_service)

initial_complaint = "My delivery was 3 days late and the box was crushed!"
msg = Content(role="user", parts=[Part(text=initial_complaint)])

print("Starting Workflow...\n")

async for event in runner.run_async(
    user_id=session.user_id,
    session_id=session.id,
    new_message=msg
):
    if isinstance(event, RequestInput):
        print("⏸️ WORKFLOW PAUSED FOR HUMAN REVIEW:\n")
        print(event.message)
        break # We hit the pause point, exit the loop
    elif event.message:
        print(f"> {event.message}")

### Execute - Phase 2: Resume Workflow
Now that the workflow is paused, we can "resume" it by calling `runner.run_async` again with the exact same `session_id`, but passing our human feedback as the `new_message`.

In [ ]:
# Try changing this to 'approve', 'reject', or custom text like 'Too formal, make it shorter.'
human_response_text = "Too formal, make it shorter and offer a 10% discount."

human_msg = Content(role="user", parts=[Part(text=human_response_text)])

print(f"▶️ RESUMING WORKFLOW WITH INPUT: '{human_response_text}'\n")

async for event in runner.run_async(
    user_id=session.user_id,
    session_id=session.id,
    new_message=human_msg
):
    if isinstance(event, RequestInput):
        print("\n⏸️ WORKFLOW PAUSED FOR HUMAN REVIEW AGAIN:\n")
        print(event.message)
        break
    elif event.message:
        print(f"> {event.message}")
    elif event.content:
        print(f"> {event.content.parts[0].text}")

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.